# Lab 2 · Lesson 2: hidden APIs and embedded data
Many websites load their data from a **hidden JSON API**. Your browser shows you that call: copy it, and you get clean JSON.

In [11]:
import json
import time

import requests
from bs4 import BeautifulSoup

USER_AGENT = "Mozilla/5.0 (compatible; ESILV-Lab-Advanced/1.0; educational use)"
session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})

## 1. Where to look first
Start at the top. Go down only if the step above does not work.

| Step | Where | Example | Stable? |
|---|---|---|---|
| 1 | an **official API** | Open Food Facts | ⭐⭐⭐ |
| 2 | a **hidden API** (the JSON the page asks for) | IKEA search | ⭐⭐ |
| 3 | **data hidden in the HTML** (JSON-LD, `__NEXT_DATA__`) | Coolblue | ⭐⭐ |
| 4 | **HTML parsing** with CSS selectors (lab 1) | any page | ⭐ |
| 5 | a **real browser** (Selenium, lab 3) | logins, pages built by JavaScript | ⭐, slow |

## 2. DevTools: find the call
On [ikea.com/fr/fr](https://www.ikea.com/fr/fr/):

1. Press `F12` (macOS: `Cmd`+`Option`+`I`) → **Network** tab.
2. Click **Fetch/XHR**. Tick **Keep log**. Click 🚫 to clear the list.
3. In **IKEA's search box** (*Que cherchez-vous ?*), type `chaise` and press Enter.
4. In the DevTools **Filter** box, type `sik.search`.
5. Click the line `search?c=sr&v=...` and look at its tabs:
   - **Headers**: the address and the method (`POST`)
   - **Payload**: what the page sent (your search)
   - **Preview**: the answer, with the products and their prices
6. Right-click that line → **Copy** → **Copy as cURL (bash)**.

> Tip: `Ctrl`+`F` in the Network tab searches inside all the answers. Search a price you see on the page.

## 3. From the copied call to a Session
Keep only the **address** and the **parameters**. Our `session` already sends our User-Agent.

In [12]:
SEARCH_URL = "https://sik.search.blue.cdtapps.com/fr/fr/search-result-page"
params = {"q": "chaise", "size": 24, "types": "PRODUCT"}

response = session.get(SEARCH_URL, params=params, timeout=20)
print(response.status_code, response.headers["Content-Type"], len(response.content), "bytes")
print(response.url)

200 application/json; charset=utf-8 224380 bytes
https://sik.search.blue.cdtapps.com/fr/fr/search-result-page?q=chaise&size=24&types=PRODUCT


## 4. Which headers are really needed?
Remove the browser's headers one by one. Each test uses a **fresh** session, so no old cookie can help.

In [13]:
BROWSER_HEADERS = {
    "accept": "*/*",
    "accept-language": "fr-FR,fr;q=0.9,en;q=0.8",
    "origin": "https://www.ikea.com",
    "referer": "https://www.ikea.com/",
    "sec-fetch-mode": "cors",
    "sec-fetch-site": "cross-site",
}


def call_with(headers):
    """A fresh session that sends our User-Agent + these headers, nothing else."""
    with requests.Session() as fresh:
        fresh.headers.update({"User-Agent": USER_AGENT, **headers})
        return fresh.get(SEARCH_URL, params=params, timeout=20)


def same_answer(a, b):
    return a.status_code == b.status_code and a.headers.get("Content-Type") == b.headers.get("Content-Type")


reference = call_with(BROWSER_HEADERS)
needed = dict(BROWSER_HEADERS)
for name in BROWSER_HEADERS:
    trial = {k: v for k, v in needed.items() if k != name}
    time.sleep(1)                                    # a pause: we are guests
    if same_answer(call_with(trial), reference):
        needed = trial                               # not needed: forget it
    else:
        print("NEEDED:", name)
print("Headers really needed:", needed or "none, only our User-Agent")

Headers really needed: none, only our User-Agent


Result: none. Our `session` is enough for every call below.

## 5. Read a JSON you do not know
Do not print it all. Print its **outline** (keys and sizes), then follow the path to the data.

In [14]:
def outline(data, depth=0, max_depth=3, max_keys=12):
    """Print the structure of a JSON: keys, types, list sizes."""
    pad = "  " * depth
    if isinstance(data, dict):
        for key, value in list(data.items())[:max_keys]:
            kind = f"{type(value).__name__}[{len(value)}]" if isinstance(value, (dict, list)) else repr(value)[:50]
            print(f"{pad}{key}: {kind}")
            if depth < max_depth:
                outline(value, depth + 1, max_depth, max_keys)
    elif isinstance(data, list) and data and depth < max_depth:
        print(f"{pad}[0]:")
        outline(data[0], depth + 1, max_depth, max_keys)


data = response.json()
outline(data["searchResultPage"], max_depth=2)

searchPhrase: 'chaise'
storeWindow: list[0]
storeCount: 0
didYouMean: list[0]
relatedSearches: list[10]
  [0]:
    count: 363
    text: 'chaise bureau'
    metadata: '612'
retiredProducts: list[0]
products: dict[6]
  badge: 112
  filters: list[14]
  sortOrders: dict[2]
    name: 'Trier'
    values: list[10]
  main: dict[5]
    items: list[24]
    start: 0
    end: 24
    max: 112
    moreToken: 'H4sIAAAAAAAA_6tWysksLlGyUgoI8ncJdQ4Jjvd19PRT0lFKz
  shelves: list[0]
  pageMessages: list[0]
content: dict[6]
  badge: 660
  filters: list[0]
  sortOrders: dict[2]
    name: 'Trier'
    values: list[0]
  main: dict[5]
    items: list[24]
    start: 0
    end: 24
    max: 660
    moreToken: 'H4sIAAAAAAAA_6tWysksLlGyUnL29wtx9QuJ93X09FPSUUrNS
  shelves: list[0]
  pageMessages: list[0]


In [15]:
first = data["searchResultPage"]["products"]["main"]["items"][0]["product"]
print(first["name"], first["typeName"], first["salesPrice"]["numeral"], first["salesPrice"]["currencyCode"])

SANDSBERG Chaise 14.99 EUR


## 6. Pagination: get all the results
APIs send results in pages. Four common ways to ask for the next page:

| Style | You send | Example |
|---|---|---|
| page number | `page=1, 2, 3...` | Open Food Facts, Shopify |
| offset | `offset=0, 20, 40...` | many APIs |
| cursor | the `next` / `cursor` value from the last answer | social networks |
| next link | the `next` URL from the last answer | GitHub |

Always: a **pause** between pages, a **maximum** number of pages, and stop on an **empty page**.

## 7. Get all the chairs: a wrong try, then the right way
IKEA found 112 chairs but sent only 24. First we try `offset`, a common way to get the next page.
This **fails on purpose**: read what the API says.

In [16]:
wrong = session.get(SEARCH_URL, params={**params, "offset": 24}, timeout=20)
print(wrong.status_code, wrong.json()["detail"])

400 Unrecognized parameter: offset


**400** = our request is wrong, and the API says why: it does not know `offset`. Do not retry a 400: fix the request.

The right way for this API: the answer gives the total (`max`), and `size` accepts big values. So ask for all of them in one call:

In [22]:
total = data["searchResultPage"]["products"]["main"]["max"]
print(total)
everything = session.get(SEARCH_URL, params={**params, "size": total}, timeout=20)
items = everything.json()["searchResultPage"]["products"]["main"]["items"]
print(everything.status_code, "|", total, "chairs exist |", len(items), "received in one call")

112
200 | 112 chairs exist | 112 received in one call


## 8. Data hidden in the HTML
Many pages carry their data as JSON inside a `<script>` tag. No browser needed: `requests` + Beautiful Soup + `json.loads`.

### 8a. JSON-LD
`<script type="application/ld+json">`: name, price, stock, rating. Shops keep it correct because Google shows it.

In [ ]:
page = session.get("https://www.coolblue.nl/en/product/935188/apple-iphone-15-128gb-black.html", timeout=30)
print(page.status_code, len(page.content), "bytes of HTML")
soup = BeautifulSoup(page.text, "lxml")

for tag in soup.select('script[type="application/ld+json"]'):
    block = json.loads(tag.string)
    print("@type:", block.get("@type"))
    if block.get("@type") == "Product":
        print("   ", block["name"], "|", block["offers"]["price"], block["offers"]["priceCurrency"], "|",
              block["offers"]["availability"], "|", block.get("aggregateRating"))

200 1633239 bytes of HTML
@type: BreadcrumbList
@type: Product
    Apple iPhone 15 128GB Black | 734 EUR | https://schema.org/OutOfStock | {'@type': 'AggregateRating', 'worstRating': 1, 'bestRating': 10, 'ratingValue': 9, 'reviewCount': 134}
@type: AggregateRating
@type: AggregateRating


1.5 MB of HTML, but the data is a few hundred bytes of JSON.
Careful: two `AggregateRating` blocks rate the **shop**, not the phone (project 3).

### 8b. `__NEXT_DATA__` (Next.js sites)
`<script id="__NEXT_DATA__">` holds **all** the page's data. Here, a small example page.

In [19]:
NEXT_PAGE = """<html><body><div id="__next">Loading...</div>
<script id="__NEXT_DATA__" type="application/json">
{"props": {"pageProps": {"product": {"id": 4242, "title": "Round Glasses", "prices": [
  {"name": "Market Price", "currency": "EUR", "price": 120}, {"name": "Sales Price", "currency": "EUR", "price": 96}],
  "stock": 0}}}, "page": "/[...slug]", "buildId": "a1b2c3d4"}
</script></body></html>"""

soup = BeautifulSoup(NEXT_PAGE, "lxml")
next_data = json.loads(soup.find("script", id="__NEXT_DATA__").string)
product = next_data["props"]["pageProps"]["product"]
print(product["title"], {p["name"]: p["price"] for p in product["prices"]}, "stock:", product["stock"])
print("Same data as pure JSON at: /_next/data/" + next_data["buildId"] + "/<page>.json")

Round Glasses {'Market Price': 120, 'Sales Price': 96} stock: 0
Same data as pure JSON at: /_next/data/a1b2c3d4/<page>.json
